# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library. We'll demonstrate accessing record sets, fields, and columns using their `@id`s, as required by the Croissant specification.

### Dataset Source
The Croissant schema source for this dataset is:
- [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

This dataset includes tabular data about cancer survivors with second primary colorectal cancer, clinicopathological and molecular characteristics, and supports analysis of MSI-H phenotype distribution. All data elements and references in this notebook use their `@id` fields as per Croissant standards.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata object (do NOT subscript or iterate)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s using the Croissant schema.

To access entities, we use the Croissant methods and refer to record sets, fields, and columns by their `@id` only.

In [ ]:
# View available record sets and their @id
record_sets = dataset.record_sets
print("Record sets available:")
record_set_ids = []
for rs in record_sets:
    print(f"  Name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description}\n")
    record_set_ids.append(rs.id)

# For the first record set, list all fields with their @id
if record_sets:
    main_rs = record_sets[0]
    print(f"Fields for RecordSet {main_rs.id}:")
    fields = main_rs.fields
    field_ids = []
    for f in fields:
        print(f"  Name: {f.name}\n  @id: {f.id}\n  DataType: {f.data_type}")
        field_ids.append(f.id)
        # If a field has columns, print column @ids
        if hasattr(f, 'columns') and f.columns:
            print("    Columns:")
            for col in f.columns:
                print(f"     - Name: {col.name}, @id: {col.id}, DataType: {col.data_type}")
    print()

## 3. Data Extraction
Load data from all record sets into Pandas DataFrames using their `@id`. All entities (record sets, fields, columns) are referenced strictly by `@id`.

In [ ]:
dataframes = {}
# Extract data for each record set using their @id
for rs in dataset.record_sets:
    rs_id = rs.id
    print(f"Loading records for RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

    print(f"Columns (@id) for {rs_id}:")
    print(df.columns.tolist())
    print(df.head(3), '\n')

# Choose the main record set for further analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None
main_df = dataframes[main_record_set_id]
print(f"Main DataFrame columns (@id): {main_df.columns.tolist()}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply basic processing: filtering, normalization, grouping. Operations reference columns by their `@id`. For demonstration, let's select a numeric field from the previously listed fields.

In [ ]:
# Identify a numeric field by @id
numeric_field_id = None
# Search for an Integer/Float field from field list
for f in dataset.record_sets[0].fields:
    if f.data_type in ('schema:Integer', 'schema:Float'):
        numeric_field_id = f.id
        print(f"Selected numeric field for EDA: {f.name} (@id={numeric_field_id})")
        break

# If no numeric field found, fallback
if numeric_field_id is None:
    numeric_field_id = main_df.select_dtypes(include=['int', 'float']).columns[0] if len(main_df.select_dtypes(include=['int', 'float']).columns) else main_df.columns[0]
    print(f"Fallback numeric field: {numeric_field_id}")

# Example threshold for filtering
threshold = 10
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Select a group field by @id (e.g., a categorical field)
group_field_id = None
for f in dataset.record_sets[0].fields:
    if f.data_type == 'schema:Text' and f.id in main_df.columns:
        group_field_id = f.id
        print(f"Grouping field: {f.name} (@id={group_field_id})")
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships between fields. All axes and legend references use the `@id` of fields.

Below, we visualize the distribution of the selected numeric field and relationship with the grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of normalized numeric field
plt.figure(figsize=(8,5))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=15, kde=True)
plt.title(f"Distribution of normalized field (@id={numeric_field_id})")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.ylabel("Count")
plt.show()

# Boxplot of numeric field grouped by categorical field
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by group (@id={group_field_id})")
    plt.xlabel(f"Group field (@id={group_field_id})")
    plt.ylabel(f"Numeric field (@id={numeric_field_id})")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
This notebook illustrated loading a clinical oncology dataset with the `mlcroissant` library using Croissant schema, accessing entities strictly by their `@id`, and performing Pandas-based EDA and visualization.

- All dataset elements (RecordSet, Field, Column) were referenced and manipulated via `@id`.
- The notebook demonstrated record extraction, field selection, filtering, normalization, grouping, and plotting using `mlcroissant`.

This approach ensures full interoperability with FAIR data systems and reproducible clinical data science workflows.